# Postprocessing Tuner
Select a model and appliance, tune the ON/OFF status hyperparameters interactively, inspect the F1 score and signal visualisation, then save the updated status back to the predictions CSV.

In [12]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.metrics import confusion_matrix

sys.path.insert(0, os.path.join('..', 'src_pytorch'))
from evaluator import compute_status
from config import PLEGMA_PARAMS

## 1 — Select model and appliance

In [17]:
model_w = widgets.Dropdown(
    options=['cnn', 'tcn'],
    value='tcn',
    description='Model:',
    style={'description_width': 'initial'},
)
appliance_w = widgets.Dropdown(
    options=['boiler', 'ac_1', 'washing_machine'],
    value='boiler',
    description='Appliance:',
    style={'description_width': 'initial'},
)
display(widgets.HBox([model_w, appliance_w]))

## 2 — Load predictions
Run this cell after selecting the model and appliance above.

In [18]:

MODEL     = model_w.value
APPLIANCE = appliance_w.value

pred_path = os.path.join('..', 'outputs', f'{MODEL}_{APPLIANCE}', 'metrics', f'{APPLIANCE}_predictions.csv')
df = pd.read_csv(pred_path)

gt   = df['ground_truth_W'].values
pred = df['prediction_W'].values

# Default hyperparameters from config
params      = PLEGMA_PARAMS[APPLIANCE]
DEF_THRESH  = int(params['threshold'])
DEF_MIN_ON  = int(params['min_on'])
DEF_MIN_OFF = int(params['min_off'])
DEF_MIN_COMMITTED = int(params.get('min_committed_duration', 0))  # 0 = not applicable

print(f"Loaded  : {pred_path}")
print(f"Samples : {len(df):,}")
print()
print(f"Default hyperparameters for '{APPLIANCE}':")
print(f"  threshold              = {DEF_THRESH} W")
print(f"  min_on                 = {DEF_MIN_ON} samples")
print(f"  min_off                = {DEF_MIN_OFF} samples")
print(f"  min_committed_duration = {DEF_MIN_COMMITTED if DEF_MIN_COMMITTED > 0 else 'not set (boiler/ac)'}")


Loaded  : ..\outputs\cnn_washing_machine\metrics\washing_machine_predictions.csv
Samples : 1,626,897

Default hyperparameters for 'washing_machine':
  threshold              = 15 W
  min_on                 = 2 samples
  min_off                = 100 samples
  min_committed_duration = 250


## 3 — Interactive hyperparameter tuning
Adjust the sliders to explore different postprocessing settings.  
Metrics and the signal plot update automatically.

In [21]:
# Shared state — the save button reads from here
_state = {}

# ── Widgets ──────────────────────────────────────────────────────────────────
style  = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

w_thresh  = widgets.IntSlider(min=1,   max=2000, step=10,  value=DEF_THRESH,  description='threshold (W)',  style=style, layout=layout)
w_min_on  = widgets.IntSlider(min=1,   max=500,  step=1,   value=DEF_MIN_ON,  description='min_on',         style=style, layout=layout)
w_min_off = widgets.IntSlider(min=1,   max=500,  step=1,   value=DEF_MIN_OFF, description='min_off',        style=style, layout=layout)
w_max_len = widgets.IntSlider(min=0,   max=1000, step=10,  value=DEF_MIN_COMMITTED, description='min_committed_duration (0=off)', style=style, layout=layout)
w_start   = widgets.IntSlider(min=0,   max=max(len(df)-500, 0), step=100, value=0,   description='Window start',  style=style, layout=layout)
w_window  = widgets.IntSlider(min=100, max=5000, step=100, value=2000,        description='Window size',    style=style, layout=layout)

out = widgets.Output()

def _update(*args):
    threshold  = w_thresh.value
    min_on     = w_min_on.value
    min_off    = w_min_off.value
    min_committed_duration = w_max_len.value if w_max_len.value > 0 else None
    start      = w_start.value
    window     = w_window.value

    gt_status   = compute_status(gt,   threshold, min_on, min_off, min_committed_duration)
    pred_status = compute_status(pred, threshold, min_on, min_off, min_committed_duration)

    # Metrics
    tn, fp, fn, tp = confusion_matrix(gt_status, pred_status, labels=[0, 1]).ravel()
    prec = tp / max(tp + fp, 1e-9)
    rec  = tp / max(tp + fn, 1e-9)
    f1   = 2 * prec * rec / max(prec + rec, 1e-9)
    acc  = (tp + tn) / (tp + tn + fp + fn)

    # Store for save button
    _state['gt_status']   = gt_status
    _state['pred_status'] = pred_status
    _state['params'] = {
        'threshold' : threshold,
        'min_on'    : min_on,
        'min_off'   : min_off,
        'min_committed_duration': min_committed_duration,
    }
    _state['metrics'] = {'f1_complex': f1, 'accuracy': acc, 'precision': prec, 'recall': rec}

    end = min(start + window, len(gt))
    t   = np.arange(start, end)

    with out:
        clear_output(wait=True)

        # ── Metrics banner ───────────────────────────────────────────────────
        print(
            f"  F1_Complex: {f1:.4f}   Accuracy: {acc:.4f}   "
            f"Precision: {prec:.4f}   Recall: {rec:.4f}"
        )

        # ── Plot ─────────────────────────────────────────────────────────────
        fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)

        # Power signals
        axes[0].plot(t, gt[start:end],   color='steelblue', lw=1,   label='Ground Truth')
        axes[0].plot(t, pred[start:end], color='darkorange', lw=1, alpha=0.85, label='Prediction')
        axes[0].axhline(threshold, color='crimson', ls='--', lw=0.9, label=f'Threshold ({threshold} W)')
        axes[0].set_ylabel('Power (W)')
        axes[0].legend(loc='upper right', fontsize=8)
        axes[0].set_title(f'{MODEL.upper()} — {APPLIANCE}  |  Power signal')

        # Status
        axes[1].fill_between(t, gt_status[start:end],   step='pre', alpha=0.55, color='steelblue',  label='GT status')
        axes[1].fill_between(t, pred_status[start:end], step='pre', alpha=0.55, color='darkorange', label='Pred status')
        axes[1].set_yticks([0, 1])
        axes[1].set_yticklabels(['OFF', 'ON'])
        axes[1].set_ylabel('Status')
        axes[1].set_xlabel('Sample index')
        axes[1].legend(loc='upper right', fontsize=8)
        axes[1].set_title(f'Duration-filtered ON/OFF  |  min_on={min_on}  min_off={min_off}  min_committed_duration={min_committed_duration}')

        plt.tight_layout()
        plt.show()

# Observe all widgets
for w in [w_thresh, w_min_on, w_min_off, w_max_len, w_start, w_window]:
    w.observe(_update, names='value')

display(
    widgets.VBox([
        widgets.HTML('<b>Postprocessing parameters</b>'),
        w_thresh, w_min_on, w_min_off, w_max_len,
        widgets.HTML('<b>Visualisation window</b>'),
        w_start, w_window,
        out,
    ])
)
_update()  # initial render

## 4 — Save selected hyperparameters
When you are satisfied with the results above, run this cell to write the updated `ground_truth_status` and `predicted_status` columns back to the predictions CSV.

In [22]:
save_btn = widgets.Button(
    description='Save status to CSV',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='220px', height='40px'),
)
save_out = widgets.Output()

def _on_save(b):
    with save_out:
        clear_output()
        if not _state:
            print('Run the tuning cell first.')
            return

        df['ground_truth_status'] = _state['gt_status']
        df['predicted_status']    = _state['pred_status']
        df['error']               = df['prediction_W'] - df['ground_truth_W']
        df['abs_error']           = df['error'].abs()
        df.to_csv(pred_path, index=False)

        p = _state['params']
        m = _state['metrics']
        print(f"Saved  → {pred_path}")
        print()
        print("Hyperparameters written:")
        print(f"  threshold              = {p['threshold']} W")
        print(f"  min_on                 = {p['min_on']} samples")
        print(f"  min_off                = {p['min_off']} samples")
        print(f"  min_committed_duration = {p['min_committed_duration']}")
        print()
        print("Metrics at save time:")
        print(f"  F1_Complex = {m['f1_complex']:.4f}")
        print(f"  Accuracy   = {m['accuracy']:.4f}")
        print(f"  Precision  = {m['precision']:.4f}")
        print(f"  Recall     = {m['recall']:.4f}")

        # Update F1_Complex in the metrics results CSV
        metrics_path = os.path.join('..', 'outputs', f'{MODEL}_{APPLIANCE}', 'metrics', f'{APPLIANCE}_results.csv')
        if os.path.exists(metrics_path):
            df_metrics = pd.read_csv(metrics_path)
            df_metrics['F1_Complex'] = round(m['f1_complex'], 4)
            df_metrics.to_csv(metrics_path, index=False)
            print(f"  F1_Complex updated in: {os.path.basename(metrics_path)}")

save_btn.on_click(_on_save)
display(save_btn, save_out)

Button(button_style='success', description='Save status to CSV', icon='check', layout=Layout(height='40px', wi…

Output()